In [13]:
import time
import pickle

from pyboolnet.trap_spaces import compute_trap_spaces

from boolmore.mask import (
    generate_source_masks,
    mask_to_sources,
)

from boolmore.phenotypes import get_mintr_for_source_comb


In [14]:
CACHE_FILE = "primes_cache.pkl"

In [15]:
with open(CACHE_FILE, "rb") as f:
    primes = pickle.load(f)
print("Loaded primes from cache.")

Loaded primes from cache.


In [16]:
def benchmark_solver_coverage(primes, source_nodes, max_output=10000):

    start = time.perf_counter()


    tr = compute_trap_spaces(primes, "min", max_output=max_output)

    elapsed = time.perf_counter() - start


    covered = set()
    for ts in tr:
        key = tuple(ts[n] for n in source_nodes)
        covered.add(key)

    return {
        "number_of_trap_spaces": len(tr),
        "elapsed": elapsed,
        "coverage": len(covered),
        "covered_set": covered,
        "coverage_per_sec": len(covered) / elapsed if elapsed > 0 else 0.0,
    }

In [17]:
def benchmark_pipeline_coverage(
    primes,
    source_nodes,
    duration=3.0,
    random_order=False,
):
    """
    Measures:
    - how many source combinations are processed
    - how many unique source-node assignments are covered
    within a fixed time budget.
    """

    start = time.perf_counter()

    number_of_trap_spaces = 0
    covered = set()

    for mask in generate_source_masks(source_nodes, {}, random_order=random_order, seed=0):

        source_comb = mask_to_sources(mask, source_nodes)

        # main computation
        tr = get_mintr_for_source_comb(primes, source_comb)

        number_of_trap_spaces += len(tr)

        # record coverage over source nodes
        key = tuple(source_comb[n] for n in source_nodes)
        covered.add(key)

        # stop after fixed time budget
        if time.perf_counter() - start >= duration:
            break

    elapsed = time.perf_counter() - start

    return {
        "number_of_trap_spaces": number_of_trap_spaces,
        "elapsed": elapsed,
        "coverage": len(covered),
        "covered_set": covered,
        "coverage_per_sec": len(covered) / elapsed if elapsed > 0 else 0.0,
    }

In [18]:
source_nodes = [
    n for n in primes
    if primes[n] == [[{n: 0}], [{n: 1}]]
]

result_A = benchmark_solver_coverage(primes, source_nodes, max_output=100000)

result_B = benchmark_pipeline_coverage(primes, source_nodes, duration=result_A["elapsed"], random_order=True)

A_set = result_A["covered_set"]
B_set = result_B["covered_set"]


for key in result_A:
    if key == "covered_set":
        continue
    print(key, result_A[key], result_B[key])

print('Jaccard similarity:', len(A_set & B_set) / len(A_set | B_set))  # Jaccard similarity

INFO there are possibly more than 100000 trap spaces.
INFO increase MaxOutput to find out.
number_of_trap_spaces 100000 32779
elapsed 55.04526351200184 55.0759889539986
coverage 96684 1148
coverage_per_sec 1756.4454020447995 20.843928938958314
Jaccard similarity: 0.00027605950616021676
